In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from model_brand_resnet import get_resnet18_brand_model
import os
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import shutil

In [ ]:
# ✅ 경로 설정
data_dir = "C:/Users/Admin/Desktop/food_brand_dataset"  # ← 사용자 경로에 맞게 수정
model_path = "best_brand_model.pth"

# ✅ 장치 설정
device = torch.device("cuda" if torch.cuda().is_available() else "cpu")

# ✅ 테스트 transform
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ✅ 테스트 데이터셋 로딩
test_dataset = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ✅ 클래스명 로드
class_names = test_dataset.classes
print("\u2705 브랜드 클래스:", class_names)

# ✅ 모델 로드
model = get_resnet18_brand_model(num_classes=len(class_names), pretrained=False).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# ✅ 예측 수행
all_preds = []
all_probs = []
all_labels = []
wrong_preds = []
wrong_paths = []
wrong_names = []

softmax = torch.nn.Softmax(dim=1)

# 원본 이미지 경로 저장용
orig_dataset = datasets.ImageFolder(os.path.join(data_dir, "test"))
image_paths = [path for path, _ in orig_dataset.imgs]

image_counter = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = softmax(outputs)
        preds = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

        for i in range(len(preds)):
            if preds[i] != labels[i]:
                idx = image_counter + i
                wrong_preds.append((images[i].cpu(), preds[i].item(), labels[i].item(), probs[i].cpu().numpy()))
                wrong_paths.append(image_paths[idx])
                wrong_names.append(f"{class_names[preds[i]]}_{class_names[labels[i]]}_{i+1:03d}.jpg")
        image_counter += len(images)

# ✅ classification report 출력
print("\n\ud83d\udcca classification report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# ✅ 전체 혼동 행렬
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
plt.figure(figsize=(8, 8))
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix (Brand Classification)")
plt.savefig("confusion_matrix_brand.png")
plt.show()

# ✅ 오답 이미지 9장 시각화 + 예측 상위 3개 확률
plt.figure(figsize=(12, 12))
for idx, (img_tensor, pred, label, prob) in enumerate(wrong_preds[:9]):
    img = img_tensor.permute(1, 2, 0).numpy()
    img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    img = np.clip(img, 0, 1)

    top3_idx = prob.argsort()[::-1][:3]
    top3_info = "\n".join([f"{class_names[i]}: {prob[i]*100:.1f}%" for i in top3_idx])

    plt.subplot(3, 3, idx + 1)
    plt.imshow(img)
    plt.title(f"True: {class_names[label]}\nPred: {class_names[pred]}\n{top3_info}")
    plt.axis("off")

plt.tight_layout()
plt.savefig("wrong_predictions.png")
plt.show()

# ✅ 오답이 자주 발생한 클래스 상위 4개 추출해서 혼동 행렬 시각화
error_counts = {}
for _, pred, label, _ in wrong_preds:
    key = (class_names[label], class_names[pred])
    error_counts[key] = error_counts.get(key, 0) + 1

sorted_errors = sorted(error_counts.items(), key=lambda x: x[1], reverse=True)
top_classes = set()
for (true_cls, pred_cls), _ in sorted_errors[:10]:
    top_classes.add(true_cls)
    top_classes.add(pred_cls)

top_classes = list(top_classes)
indices = [class_names.index(cls) for cls in top_classes]

filtered_cm = cm[np.ix_(indices, indices)]
filtered_labels = [class_names[i] for i in indices]

plt.figure(figsize=(6, 6))
ConfusionMatrixDisplay(confusion_matrix=filtered_cm, display_labels=filtered_labels).plot(cmap=plt.cm.Oranges, xticks_rotation=45)
plt.title("Confusion Matrix (Top Confused Classes)")
plt.savefig("confusion_matrix_top_confused.png")
plt.show()

# ✅ 오답 이미지 복사 저장
wrong_dir = "wrong_images"
os.makedirs(wrong_dir, exist_ok=True)

for i, (wrong_path, wrong_name) in enumerate(zip(wrong_paths, wrong_names)):
    dest_path = os.path.join(wrong_dir, wrong_name)
    shutil.copy(wrong_path, dest_path)

print(f"\n✅ 오답 이미지 {len(wrong_paths)}장을 '{wrong_dir}' 폴더에 예측_실제 형식으로 저장 완료했습니다.")


# 상위 예측 3개 브랜드 + 확률 표현
# 브랜드 혼동 행렬 시각화 이미지 :confusion_matrix_brand.png
# 오답 이미지 9장 시각화 : wrong_predictions.png
# precision, reall,f1-score,accuracy 등 전체 지표 : classification_report
# 자주 틀린 클래스만 혼동 행렬 : confusion_matrix_top_confused.png
# 오답 이미지만 따로 폴더 저장 wrong_images/ 